# Future regular season games (Current date - 9/27/2026)

future_games

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW future_games AS

-- Grabbing the future game details from the bronze layer's JSON
WITH exploded_games AS (

    SELECT
        game.gamePk AS game_pk,

        CAST(
            game.officialDate
            AS DATE
        ) AS game_date,

        game.gameType AS game_type,
        game.teams.away.team.id AS away_team_id,
        game.teams.home.team.id AS home_team_id,

        game.status.detailedState AS game_status,

        s.ingestion_timestamp

    FROM bronze.mlb_schedule_raw s

    LATERAL VIEW EXPLODE(
        FROM_JSON(
            GET_JSON_OBJECT(
                s.raw_json,
                '$.dates[0].games'
            ),

            'array<struct<
                gamePk:bigint,
                gameDate:string,
                officialDate:string,
                gameType:string,
                status:struct<detailedState:string>,
                teams:struct<
                    away:struct<
                        team:struct<id:bigint>
                    >,
                    home:struct<
                        team:struct<id:bigint>
                    >
                >
            >>'
        )
    ) exploded AS game

    -- Regualar season only
    WHERE game.gameType = 'R'

        AND game.status.detailedState IN (
            'Scheduled',
            'Pre-Game'
        )

        --Ignore old schedule snapshots
        AND CAST(game.officialDate AS DATE) >= current_date()
),

-- finding duplicate future games (if any)
deduplicated AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY game_date, home_team_id, away_team_id
            ORDER BY ingestion_timestamp DESC
        ) AS rn
    FROM exploded_games
)

-- Game details from latest future schedule snapshot
-- matching team_ids with team name
SELECT
    f.game_pk,
    f.game_date,
    f.game_type,
    f.away_team_id,
    away.team_name AS away_team,
    home_team_id,
    home.team_name AS home_team,
    f.game_status

FROM deduplicated f
LEFT JOIN silver.mlb_teams away
ON f.away_team_id = away.team_id
LEFT JOIN silver.mlb_teams home
ON f.home_team_id = home.team_id

WHERE rn = 1;

## Checking for Number of remaining games left for each team

In [0]:
%sql
-- # of games remaining for each team
WITH remaining AS (

    SELECT
        team_name,
        COUNT(*) AS schedule_remaining
    FROM (

        SELECT away_team AS team_name
        FROM future_games

        UNION ALL

        SELECT home_team AS team_name
        FROM future_games

    ) t

    GROUP BY team_name
),

standings_check AS (

    SELECT
        team_name,
        wins,
        losses,
        162 - wins - losses AS expected_remaining
    FROM gold.mlb_standings
)

SELECT
    s.team_name,
    s.wins,
    s.losses,
    s.expected_remaining,
    COALESCE(r.schedule_remaining, 0) AS schedule_remaining,
    COALESCE(r.schedule_remaining, 0)
        - s.expected_remaining AS difference

FROM standings_check s

LEFT JOIN remaining r
    ON s.team_name = r.team_name

ORDER BY difference DESC;

# Future Games Win Probability (based on current win %)

team_win_pct

In [0]:
%sql
-- Calculating each team's current win percentage
CREATE OR REPLACE TEMP VIEW team_win_pct AS

SELECT
    team_name,
    Wins,
    Losses,
    home_wins,
    home_losses,
    away_wins,
    away_losses,

    ROUND(
        Wins / (Wins + Losses),
        4
    ) AS win_pct

FROM gold.mlb_standings;

future_game_probabilities (Each games win probability)

In [0]:
%sql
-- Calculating each future game based on each team's current win percentage
CREATE OR REPLACE TEMP VIEW future_game_probabilities AS

SELECT
    f.game_pk,
    f.game_date,

    f.away_team,
    f.home_team,

    aw.win_pct AS away_current_win_pct,
    hw.win_pct AS home_current_win_pct,

    -- Relative team strength
    -- future Win probability per game based on current win pct
    (
        hw.win_pct /
        (hw.win_pct + aw.win_pct)
    ) AS home_current_game_probability,

    (
        aw.win_pct /
        (hw.win_pct + aw.win_pct)
    ) AS away_current_game_probability,

    -- Caps the home win probability at 99%
    ROUND(LEAST(
        0.99,
        (
            home_current_win_pct /
            (home_current_win_pct + away_current_win_pct)
        ) + 0.03  -- home field advantage
    ), 2) AS home_current_game_win_probability,

    -- Floors the home win probability at 99%
    ROUND(GREATEST(
        0.01,
        1 -
        (
            (
                home_current_win_pct /
                (home_current_win_pct + away_current_win_pct)
            ) + 0.03
        )
    ), 2) AS away_current_game_win_probability



FROM future_games f

LEFT JOIN team_win_pct aw
    ON f.away_team = aw.team_name

LEFT JOIN team_win_pct hw
    ON f.home_team = hw.team_name;


SELECT
    game_date,
    away_team,
    home_team,
    away_current_game_win_probability AS away_probability,
    home_current_game_win_probability AS home_probability
FROM future_game_probabilities
ORDER BY game_date, game_pk;

future_games_df

In [0]:
# Same table as future_game_probabilites above but in Python
future_games_df = spark.sql("""
    SELECT
        game_pk,
        game_date,
        away_team,
        home_team,
        away_current_game_win_probability AS away_probability,
        home_current_game_win_probability AS home_probability
    FROM future_game_probabilities
    WHERE away_current_game_win_probability IS NOT NULL
      AND home_current_game_win_probability IS NOT NULL
""")

###Writing the future_games_df probabilites into Gold Layer

In [0]:
future_games_df.write.mode("overwrite").saveAsTable(
    "gold.mlb_game_probabilities"
)

standings_df

In [0]:
# Getting standings table from gold layer
standings_df = spark.sql("""
    SELECT
        team_name,
        wins,
        losses
    FROM gold.mlb_standings
""")

display(standings_df)

In [0]:
games = future_games_df.collect()
standings_rows = standings_df.collect()

In [0]:
starting_records = {
    row["team_name"]: {
        "wins": row["wins"],
        "losses": row["losses"]
    }
    for row in standings_rows
}

# 10,000 Simulations of the future schedule

In [0]:
import random
import copy

num_simulations = 10000

# Final standings
all_simulations = []

# Every individual game outcome
simulation_game_results = []


for sim in range(num_simulations):

    # Start from original standings
    sim_records = copy.deepcopy(starting_records)

    # Simulate every remaining game
    for game_number, game in enumerate(games, start=1):

        away = game["away_team"]
        home = game["home_team"]

        away_prob = game["away_probability"]

        # Generate random outcome between 0.0 and 1.0
        if random.random() < away_prob:

            winner = away
            loser = home

        else:

            winner = home
            loser = away


        # Update records
        sim_records[winner]["wins"] += 1
        sim_records[loser]["losses"] += 1


        # SAVE THE ACTUAL GAME RESULT
        simulation_game_results.append({

            "simulation": sim + 1,

            "game_number": game_number,

            "game_pk": game["game_pk"],

            "game_date": game["game_date"],

            "away_team": away,

            "home_team": home,

            "winner": winner,

            "loser": loser,

            "away_probability": away_prob

        })


    # Save final standings
    for team, record in sim_records.items():

        all_simulations.append({

            "simulation": sim + 1,

            "team": team,

            "wins": record["wins"],

            "losses": record["losses"]

        })


print(f"Completed {num_simulations:,} simulations")
print(f"Final standings rows: {len(all_simulations):,}")
print(f"Game result rows: {len(simulation_game_results):,}")

simulations_df (in table form)

In [0]:
import pyspark.sql.functions as F

# Each team's record for each simulation (10,000 sims for each team) in table form
simulations_df = spark.createDataFrame(all_simulations)

simulations_df = simulations_df.select(
    F.col("simulation").alias("simulation"),
    F.col("team").alias("team"),
    "wins",
    "losses"
)
display(simulations_df)

### Simulation checks: total sims (10,000), total teams (30), total rows (30,000)

In [0]:
# Check number of simulations
print("Simulations:", simulations_df.select("simulation").distinct().count())

# Check number of teams
print("Teams:", simulations_df.select("team").distinct().count())

# Check total rows
print("Rows:", simulations_df.count())

##Writing the simulations_df (all simulation results) into Gold Layer

In [0]:
simulations_df = spark.createDataFrame(all_simulations)

simulations_df.write.mode("overwrite").saveAsTable(
    "gold.mlb_simulations"
)

simulation_game_results (table)

In [0]:
# putting simulation_game_results into table form
simulation_game_results_df = spark.createDataFrame(
    simulation_game_results
)

simulation_game_results_df.createOrReplaceTempView(
    "simulation_game_results"
)

display(simulation_game_results_df.limit(20))

final_wins_by_team

In [0]:
from collections import defaultdict

# Starting wins for every team
starting_wins = {
    team: record["wins"]
    for team, record in starting_records.items()
}


# ==========================================
# CALCULATE FINAL WINS FOR EVERY
# TEAM IN EVERY SIMULATION
# ==========================================

final_wins_by_team = defaultdict(
    lambda: defaultdict(int)
)

# Start every simulation with the team's current wins
for team, wins in starting_wins.items():

    for sim in range(1, num_simulations + 1):

        final_wins_by_team[team][sim] = wins


# Add simulated wins
for row in simulation_game_results:

    winner = row["winner"]
    sim = row["simulation"]

    if winner in final_wins_by_team:

        final_wins_by_team[winner][sim] += 1


print("Finished calculating final wins.")

selected_team_simulations (min, max, median with matched simulation)

In [0]:
# ==========================================
# FIND PESSIMISTIC / BASE / OPTIMISTIC
# SIMULATION FOR EVERY TEAM
# ==========================================

selected_team_simulations = {}

for team in starting_wins.keys():

    results = sorted(
        final_wins_by_team[team].items(),
        key=lambda x: x[1]
    )

    # Minimum
    min_simulation, min_wins = results[0]

    # Median
    median_index = len(results) // 2
    median_simulation, median_wins = results[median_index]

    # Maximum
    max_simulation, max_wins = results[-1]

    selected_team_simulations[team] = {

        "Pessimistic": {
            "simulation": min_simulation,
            "final_wins": min_wins
        },

        "Base": {
            "simulation": median_simulation,
            "final_wins": median_wins
        },

        "Optimistic": {
            "simulation": max_simulation,
            "final_wins": max_wins
        }
    }

###Checking selected_team_simulations

In [0]:
for team in selected_team_simulations:

    print(
        f"{team}: "
        f"Pessimistic = {selected_team_simulations[team]['Pessimistic']['final_wins']} "
        f"(Sim {selected_team_simulations[team]['Pessimistic']['simulation']}), "
        f"Base = {selected_team_simulations[team]['Base']['final_wins']} "
        f"(Sim {selected_team_simulations[team]['Base']['simulation']}), "
        f"Optimistic = {selected_team_simulations[team]['Optimistic']['final_wins']} "
        f"(Sim {selected_team_simulations[team]['Optimistic']['simulation']})"
    )

projected_paths

In [0]:
# ==========================================
# BUILD THREE SIMULATED PATHS Game-by-Game results FOR
# EVERY TEAM
# ==========================================

projection_paths = []


# Create quick lookup:
# (team, simulation) -> scenario(s)

selected_lookup = defaultdict(list)

for team, scenarios in selected_team_simulations.items():

    for scenario, info in scenarios.items():

        selected_lookup[
            (team, info["simulation"])
        ].append(scenario)


# Process every simulated game
for row in simulation_game_results:

    simulation = row["simulation"]

    away = row["away_team"]
    home = row["home_team"]

    # Check both teams because either one
    # could have one of its three selected simulations
    for team in [away, home]:

        key = (team, simulation)

        if key not in selected_lookup:
            continue

        # Determine whether this is home or away
        if team == away:
            is_winner = row["winner"] == away
        else:
            is_winner = row["winner"] == home

        for scenario in selected_lookup[key]:

            projection_paths.append({
                "team": team,
                "scenario": scenario,
                "simulation": simulation,
                "game_pk": row["game_pk"],
                "game_date": row["game_date"],
                "game_number": None,
                "away_team": away,
                "home_team": home,
                "winner": row["winner"],
                "won_game": 1 if is_winner else 0
            })


print(
    f"Created {len(projection_paths):,} projection rows."
)

Calculating the projected wins

In [0]:
# ==========================================
# CALCULATE CUMULATIVE PROJECTED WINS
# ==========================================

# Sort chronologically
projection_paths = sorted(
    projection_paths,
    key=lambda x: (
        x["team"],
        x["scenario"],
        x["game_date"],
        x["game_pk"]
    )
)


current_team = None
current_scenario = None
current_wins = None
game_number = 0


for row in projection_paths:

    # New team/scenario combination
    if (
        row["team"] != current_team
        or row["scenario"] != current_scenario
    ):

        current_team = row["team"]
        current_scenario = row["scenario"]

        current_wins = starting_wins[current_team]

        game_number = 0

    game_number += 1

    current_wins += row["won_game"]

    row["game_number"] = game_number
    row["projected_wins"] = current_wins

all_team_projection_df (table)

In [0]:
all_team_projection_df = spark.createDataFrame(
    projection_paths
)

display(
    all_team_projection_df
    .orderBy(
        "team",
        "game_date",
        "scenario"
    )
)

###Writing all_team_projected_df into Gold Layer

In [0]:
all_team_projection_df.write \
    .mode("overwrite") \
    .saveAsTable(
        "gold.mlb_team_simulation_projection"
    )

## Checking by looking at one team (Milwaukee Brewers)

In [0]:
brewers_check = [
    x for x in projection_paths
    if x["team"] == "Milwaukee Brewers"
]

for scenario in ["Pessimistic", "Base", "Optimistic"]:

    results = [
        x for x in brewers_check
        if x["scenario"] == scenario
    ]

    print(
        scenario,
        "→",
        results[-1]["projected_wins"],
        "wins",
        "| Simulation:",
        results[0]["simulation"]
    )

In [0]:
%sql
SELECT 
    * 
FROM gold.mlb_team_simulation_projection
WHERE team = 'Milwaukee Brewers'

#Testing with one team only (Milwaukee Brewers)

In [0]:
# brewers_results = [
#     x for x in simulation_game_results
#     if x["away_team"] == "Milwaukee Brewers"
#     or x["home_team"] == "Milwaukee Brewers"
# ]

# print(f"Brewers simulated games: {len(brewers_results):,}")

In [0]:
# from collections import defaultdict

# brewers_wins = defaultdict(int)

# for row in brewers_results:

#     if row["winner"] == "Milwaukee Brewers":
#         brewers_wins[row["simulation"]] += 1

In [0]:
# brewers_starting_wins = starting_records["Milwaukee Brewers"]["wins"]

# brewers_final_wins = {
#     sim: brewers_starting_wins + wins
#     for sim, wins in brewers_wins.items()
# }

In [0]:
# print("Minimum:", min(brewers_final_wins.values()))
# print("Median:", sorted(brewers_final_wins.values())[len(brewers_final_wins)//2])
# print("Maximum:", max(brewers_final_wins.values()))

In [0]:
# sorted_brewers = sorted(
#     brewers_final_wins.items(),
#     key=lambda x: x[1]
# )

# # Minimum
# min_simulation = sorted_brewers[0]

# # Maximum
# max_simulation = sorted_brewers[-1]

# # Median
# median_index = len(sorted_brewers) // 2
# median_simulation = sorted_brewers[median_index]

# print("Pessimistic:", min_simulation)
# print("Base:", median_simulation)
# print("Optimistic:", max_simulation)

In [0]:
# pessimistic_sim = min_simulation[0]
# pessimistic_wins = min_simulation[1]

# base_sim = median_simulation[0]
# base_wins = median_simulation[1]

# optimistic_sim = max_simulation[0]
# optimistic_wins = max_simulation[1]

# print(
#     f"Pessimistic: Simulation {pessimistic_sim} → {pessimistic_wins} wins"
# )

# print(
#     f"Base: Simulation {base_sim} → {base_wins} wins"
# )

# print(
#     f"Optimistic: Simulation {optimistic_sim} → {optimistic_wins} wins"
# )

In [0]:
# selected_simulations = {
#     pessimistic_sim: "Pessimistic",
#     base_sim: "Base",
#     optimistic_sim: "Optimistic"
# }

# brewers_chart_data = []

# for row in brewers_results:

#     if row["simulation"] in selected_simulations:

#         brewers_chart_data.append({
#             "simulation": row["simulation"],
#             "game_pk": row["game_pk"],
#             "game_date": row["game_date"],
#             "away_team": row["away_team"],
#             "home_team": row["home_team"],
#             "scenario": selected_simulations[row["simulation"]],
#             "winner": row["winner"]
#         })

In [0]:
# # The three simulations we selected
# selected_simulations = {
#     971: "Pessimistic",
#     7643: "Base",
#     8475: "Optimistic"
# }

# # Starting wins for Milwaukee
# starting_wins = starting_records["Milwaukee Brewers"]["wins"]

# # Get only Milwaukee games from the three selected simulations
# brewers_selected_games = []

# for row in simulation_game_results:

#     if (
#         row["simulation"] in selected_simulations
#         and (
#             row["away_team"] == "Milwaukee Brewers"
#             or row["home_team"] == "Milwaukee Brewers"
#         )
#     ):

#         brewers_selected_games.append({
#             "simulation": row["simulation"],
#             "scenario": selected_simulations[row["simulation"]],
#             "game_pk": row["game_pk"],
#             "game_date": row["game_date"],
#             "away_team": row["away_team"],
#             "home_team": row["home_team"],
#             "winner": row["winner"]
#         })

# print(f"Selected Brewers games: {len(brewers_selected_games):,}")

In [0]:
# # Sort the games chronologically within each simulation
# brewers_selected_games = sorted(
#     brewers_selected_games,
#     key=lambda x: (
#         x["simulation"],
#         x["game_date"],
#         x["game_pk"]
#     )
# )

# # Calculate cumulative wins
# current_simulation = None
# current_wins = starting_wins
# game_number = 0

# brewers_projection_paths = []

# for row in brewers_selected_games:

#     # Reset when moving to a new simulation
#     if row["simulation"] != current_simulation:

#         current_simulation = row["simulation"]
#         current_wins = starting_wins
#         game_number = 0

#     game_number += 1

#     # Did Milwaukee win?
#     if row["winner"] == "Milwaukee Brewers":
#         current_wins += 1

#     brewers_projection_paths.append({
#         "simulation": row["simulation"],
#         "scenario": row["scenario"],
#         "game_number": game_number,
#         "game_pk": row["game_pk"],
#         "game_date": row["game_date"],
#         "away_team": row["away_team"],
#         "home_team": row["home_team"],
#         "winner": row["winner"],
#         "projected_wins": current_wins
#     })

# print(f"Projection rows: {len(brewers_projection_paths):,}")

In [0]:
# for scenario in ["Pessimistic", "Base", "Optimistic"]:

#     results = [
#         x for x in brewers_projection_paths
#         if x["scenario"] == scenario
#     ]

#     print(
#         scenario,
#         "→",
#         results[-1]["projected_wins"],
#         "wins"
#     )

In [0]:
# display(
#     spark.createDataFrame(
#         brewers_projection_paths
#     ).orderBy(
#         "game_date",
#         "game_pk",
#         "scenario"
#     )
# )

## Team Metadata dictionary for League and Division

team_info

In [0]:
# Creating a team metadata dictionary
team_info = {}

for row in spark.table("silver.mlb_teams").collect():
    team_info[row["team_name"]] = {
        "league": row["league_name"],
        "division": row["division_name"]
    }

#Playoff Predictions

### Using One Simulation for starters

In [0]:
# Using 1st simulation only (for right now)
# sim1 = (
#    simulations_df
#    .filter(F.col("simulation") == 1)
#    .collect()
# )

# sim1_standings = []

# for row in sim1:

#     team = row["team"]

#     sim1_standings.append({
#         "team": team,
#         "wins": row["wins"],
#         "losses": row["losses"],
#         "league": team_info[team]["league"],
#         "division": team_info[team]["division"]
#     })

division_winners

In [0]:
# Determining Division Winners
# division_winners = {}

# for league in ["American League", "National League"]:

#     league_teams = [
#         x for x in sim1_standings
#         if x["league"] == league
#     ]

#     divisions = set(
#         x["division"]
#         for x in league_teams
#     )

#     for division in divisions:

#         division_teams = [
#             x for x in league_teams
#             if x["division"] == division
#         ]

#         winner = max(
#             division_teams,
#             key=lambda x: x["wins"]
#         )

#         division_winners[winner["team"]] = winner

# division_winners

wild card candidates list

In [0]:
# Removing division winners to get wildcard teams
# division_winner_names = set(division_winners.keys())

# wild_card_candidates = [
#     x for x in sim1_standings
#     if x["team"] not in division_winner_names
# ]

wild_cards

In [0]:
# Selecting the top three teams from each league
# wild_cards = {}

# for league in ["American League", "National League"]:

#     candidates = [
#         x for x in wild_card_candidates
#         if x["league"] == league
#     ]

#     candidates = sorted(
#         candidates,
#         key=lambda x: x["wins"],
#         reverse=True
#     )

#     wild_cards[league] = candidates[:3]

# wild_cards

list of playoff teams (6 division winners, 6 wild cards)

In [0]:
# Combining the division winners and wild cards
# playoff_teams = set(division_winners.keys())

# for league in ["American League", "National League"]:
#     for team in wild_cards[league]:
#         playoff_teams.add(team["team"])

# len(playoff_teams), playoff_teams

##Using 10,000 Simulations Now

In [0]:
# ==========================================
# PREPARE ALL SIMULATIONS
# ==========================================

all_sim_rows = simulations_df.collect()

# Group rows by simulation
simulations = {}

for row in all_sim_rows:

    sim = row["simulation"]

    if sim not in simulations:
        simulations[sim] = []

    team = row["team"]

    simulations[sim].append({
        "team": team,
        "wins": row["wins"],
        "losses": row["losses"],
        "league": team_info[team]["league"],
        "division": team_info[team]["division"]
    })


# ==========================================
# COUNTERS
# ==========================================

playoff_counts = {}
division_winner_counts = {}
wild_card_counts = {}


# ==========================================
# PROCESS EACH SIMULATION
# ==========================================

for sim, standings in simulations.items():

    # --------------------------------------
    # DIVISION WINNERS
    # --------------------------------------

    division_winners = {}

    for league in ["American League", "National League"]:

        league_teams = [
            x for x in standings
            if x["league"] == league
        ]

        divisions = set(
            x["division"]
            for x in league_teams
        )

        for division in divisions:

            division_teams = [
                x for x in league_teams
                if x["division"] == division
            ]

            winner = max(
                division_teams,
                key=lambda x: x["wins"]
            )

            division_winners[winner["team"]] = winner


    # --------------------------------------
    # REMOVE DIVISION WINNERS
    # --------------------------------------

    division_winner_names = set(
        division_winners.keys()
    )

    wild_card_candidates = [
        x for x in standings
        if x["team"] not in division_winner_names
    ]


    # --------------------------------------
    # WILD CARDS
    # --------------------------------------

    wild_cards = {}

    for league in ["American League", "National League"]:

        candidates = [
            x for x in wild_card_candidates
            if x["league"] == league
        ]

        candidates = sorted(
            candidates,
            key=lambda x: x["wins"],
            reverse=True
        )

        wild_cards[league] = candidates[:3]


    # --------------------------------------
    # COUNT DIVISION WINNERS
    # --------------------------------------

    for team in division_winners:

        division_winner_counts[team] = (
            division_winner_counts.get(team, 0) + 1
        )

        playoff_counts[team] = (
            playoff_counts.get(team, 0) + 1
        )


    # --------------------------------------
    # COUNT WILD CARDS
    # --------------------------------------

    for league in wild_cards:

        for team in wild_cards[league]:

            team_name = team["team"]

            wild_card_counts[team_name] = (
                wild_card_counts.get(team_name, 0) + 1
            )

            playoff_counts[team_name] = (
                playoff_counts.get(team_name, 0) + 1
            )

playoff_results

In [0]:
# ==========================================
# CALCULATE PROBABILITIES
# ==========================================

playoff_results = []

for team in team_info.keys():

    playoff_results.append({

        "team": team,

        "playoff_probability":
            playoff_counts.get(team, 0) / num_simulations,

        "division_winner_probability":
            division_winner_counts.get(team, 0) / num_simulations,

        "wild_card_probability":
            wild_card_counts.get(team, 0) / num_simulations
    })


# Sort by playoff probability
playoff_results = sorted(
    playoff_results,
    key=lambda x: x["playoff_probability"],
    reverse=True
)

In [0]:
# printing playoff probabilites for every team
for x in playoff_results:

    print(
        f"{x['team']:<25}"
        f"Playoffs: {x['playoff_probability']:.1%}  "
        f"Division: {x['division_winner_probability']:.1%}  "
        f"Wild Card: {x['wild_card_probability']:.1%}"
    )

playoff_results_df (table)

In [0]:
# putting playoff_results into table form
playoff_results_df = spark.createDataFrame(playoff_results)

display(
    playoff_results_df
    .orderBy(F.col("playoff_probability").desc())
)

In [0]:
# rounding percentages
from pyspark.sql.functions import round

playoff_results_df = (
    playoff_results_df
    .withColumn(
        "playoff_probability",
        round(F.col("playoff_probability") * 100, 2)
    )
    .withColumn(
        "division_winner_probability",
        round(F.col("division_winner_probability") * 100, 2)
    )
    .withColumn(
        "wild_card_probability",
        round(F.col("wild_card_probability") * 100, 2)
    )
)

playoff_results_df = playoff_results_df.select(
    "team",
    "playoff_probability",
    "division_winner_probability",
    "wild_card_probability"
)

display(
    playoff_results_df
    .orderBy(F.col("playoff_probability").desc())
)

###Writing playoff_results_df into Gold Layer (for later use)

In [0]:
playoff_results_df.write.mode("overwrite").saveAsTable(
    "gold.mlb_playoff_predictions"
)

# Seeding Probabilities

In [0]:
# ==========================================
# PLAYOFF SEEDING COUNTERS
# ==========================================

seed_counts = {
    team: {
        1: 0,
        2: 0,
        3: 0,
        4: 0,
        5: 0,
        6: 0
    }
    for team in team_info.keys()
}


# ==========================================
# PROCESS EACH SIMULATION
# ==========================================

for sim, standings in simulations.items():

    for league in ["American League", "National League"]:

        league_teams = [
            x for x in standings
            if x["league"] == league
        ]


        # --------------------------------------
        # FIND DIVISION WINNERS
        # --------------------------------------

        division_winners = []

        divisions = set(
            x["division"]
            for x in league_teams
        )

        for division in divisions:

            division_teams = [
                x for x in league_teams
                if x["division"] == division
            ]

            winner = max(
                division_teams,
                key=lambda x: x["wins"]
            )

            division_winners.append(winner)


        # --------------------------------------
        # SEEDS 1-3
        # --------------------------------------

        division_winners = sorted(
            division_winners,
            key=lambda x: x["wins"],
            reverse=True
        )

        for seed, team in enumerate(
            division_winners,
            start=1
        ):

            seed_counts[team["team"]][seed] += 1


        # --------------------------------------
        # REMOVE DIVISION WINNERS
        # --------------------------------------

        division_winner_names = {
            x["team"]
            for x in division_winners
        }

        wild_card_candidates = [
            x for x in league_teams
            if x["team"] not in division_winner_names
        ]


        # --------------------------------------
        # SEEDS 4-6
        # --------------------------------------

        wild_card_candidates = sorted(
            wild_card_candidates,
            key=lambda x: x["wins"],
            reverse=True
        )

        for seed, team in enumerate(
            wild_card_candidates[:3],
            start=4
        ):

            seed_counts[team["team"]][seed] += 1

seed_results

In [0]:
seed_results = []

for team in team_info.keys():

    result = {
        "team": team,
        "league": team_info[team]["league"],
        "division": team_info[team]["division"]
    }

    for seed in range(1, 7):

        result[f"seed_{seed}_probability"] = (
            seed_counts[team][seed] / num_simulations
        )

    seed_results.append(result)

seed_results_df (table)

In [0]:
# putting seed_results into table form
seed_results_df = spark.createDataFrame(seed_results)

display(
    seed_results_df
    .orderBy(
        F.col("league"),
        F.col("seed_1_probability").desc()
    )
)

###Writing into Gold Layer (for later use)

In [0]:
seed_results_df.write.mode("overwrite").saveAsTable(
    "gold.mlb_seed_predictions"
)

In [0]:
# import random
# import copy

# # ==========================================
# # DYNAMIC PROBABILITY SIMULATION
# # ==========================================

# num_simulations_dynamic = 10000

# all_dynamic_simulations = []


# for sim in range(num_simulations_dynamic):

#     # --------------------------------------
#     # Start from actual current standings
#     # --------------------------------------

#     sim_records = copy.deepcopy(starting_records)


#     # --------------------------------------
#     # Simulate each remaining game
#     # --------------------------------------

#     for game in games:

#         away = game["away_team"]
#         home = game["home_team"]


#         # ==================================
#         # CALCULATE CURRENT WIN PERCENTAGES
#         # ==================================

#         away_wins = sim_records[away]["wins"]
#         away_losses = sim_records[away]["losses"]

#         home_wins = sim_records[home]["wins"]
#         home_losses = sim_records[home]["losses"]


#         away_win_pct = (
#             away_wins /
#             (away_wins + away_losses)
#         )

#         home_win_pct = (
#             home_wins /
#             (home_wins + home_losses)
#         )


#         # ==================================
#         # CALCULATE GAME PROBABILITY
#         # ==================================

#         # Relative team strength
#         away_probability = (
#             away_win_pct /
#             (away_win_pct + home_win_pct)
#         )

#         home_probability = (
#             home_win_pct /
#             (away_win_pct + home_win_pct)
#         )


#         # ==================================
#         # ADD HOME FIELD ADVANTAGE
#         # ==================================

#         home_probability = home_probability + 0.03


#         # Keep probabilities between 1% and 99%

#         home_probability = min(
#             0.99,
#             max(0.01, home_probability)
#         )

#         away_probability = 1 - home_probability


#         # ==================================
#         # SIMULATE GAME
#         # ==================================

#         if random.random() < away_probability:

#             # Away team wins
#             sim_records[away]["wins"] += 1
#             sim_records[home]["losses"] += 1

#         else:

#             # Home team wins
#             sim_records[home]["wins"] += 1
#             sim_records[away]["losses"] += 1


#     # ==========================================
#     # SAVE FINAL STANDINGS
#     # ==========================================

#     for team, record in sim_records.items():

#         all_dynamic_simulations.append({

#             "simulation": sim + 1,

#             "team": team,

#             "wins": record["wins"],

#             "losses": record["losses"]

#         })


# print(
#     f"Completed {num_simulations_dynamic:,} dynamic simulations"
# )

# print(
#     f"Total rows: {len(all_dynamic_simulations):,}"
# )

In [0]:
# dynamic_simulations_df = spark.createDataFrame(
#     all_dynamic_simulations
# )

# display(dynamic_simulations_df)

In [0]:
# # ==========================================
# # PREPARE DYNAMIC SIMULATIONS
# # ==========================================

# all_dynamic_rows = dynamic_simulations_df.collect()

# dynamic_simulations = {}

# for row in all_dynamic_rows:

#     sim = row["simulation"]

#     if sim not in dynamic_simulations:
#         dynamic_simulations[sim] = []

#     team = row["team"]

#     dynamic_simulations[sim].append({

#         "team": team,

#         "wins": row["wins"],

#         "losses": row["losses"],

#         "league": team_info[team]["league"],

#         "division": team_info[team]["division"]

#     })


# # ==========================================
# # COUNTERS
# # ==========================================

# dynamic_playoff_counts = {}

# dynamic_division_winner_counts = {}

# dynamic_wild_card_counts = {}


# # ==========================================
# # PROCESS EACH SIMULATION
# # ==========================================

# for sim, standings in dynamic_simulations.items():

#     # --------------------------------------
#     # DIVISION WINNERS
#     # --------------------------------------

#     division_winners = {}

#     for league in [
#         "American League",
#         "National League"
#     ]:

#         league_teams = [
#             x for x in standings
#             if x["league"] == league
#         ]

#         divisions = set(
#             x["division"]
#             for x in league_teams
#         )

#         for division in divisions:

#             division_teams = [
#                 x for x in league_teams
#                 if x["division"] == division
#             ]

#             winner = max(
#                 division_teams,
#                 key=lambda x: x["wins"]
#             )

#             division_winners[winner["team"]] = winner


#     # --------------------------------------
#     # REMOVE DIVISION WINNERS
#     # --------------------------------------

#     division_winner_names = set(
#         division_winners.keys()
#     )

#     wild_card_candidates = [
#         x for x in standings
#         if x["team"] not in division_winner_names
#     ]


#     # --------------------------------------
#     # WILD CARDS
#     # --------------------------------------

#     wild_cards = {}

#     for league in [
#         "American League",
#         "National League"
#     ]:

#         candidates = [
#             x for x in wild_card_candidates
#             if x["league"] == league
#         ]

#         candidates = sorted(
#             candidates,
#             key=lambda x: x["wins"],
#             reverse=True
#         )

#         wild_cards[league] = candidates[:3]


#     # --------------------------------------
#     # COUNT DIVISION WINNERS
#     # --------------------------------------

#     for team in division_winners:

#         dynamic_division_winner_counts[team] = (
#             dynamic_division_winner_counts.get(team, 0) + 1
#         )

#         dynamic_playoff_counts[team] = (
#             dynamic_playoff_counts.get(team, 0) + 1
#         )


#     # --------------------------------------
#     # COUNT WILD CARDS
#     # --------------------------------------

#     for league in wild_cards:

#         for team in wild_cards[league]:

#             team_name = team["team"]

#             dynamic_wild_card_counts[team_name] = (
#                 dynamic_wild_card_counts.get(team_name, 0) + 1
#             )

#             dynamic_playoff_counts[team_name] = (
#                 dynamic_playoff_counts.get(team_name, 0) + 1
#             )

In [0]:
# # ==========================================
# # DYNAMIC PROBABILITY RESULTS
# # ==========================================

# dynamic_playoff_results = []

# for team in team_info.keys():

#     dynamic_playoff_results.append({

#         "team": team,

#         "playoff_probability":
#             dynamic_playoff_counts.get(
#                 team, 0
#             ) / num_simulations_dynamic,

#         "division_winner_probability":
#             dynamic_division_winner_counts.get(
#                 team, 0
#             ) / num_simulations_dynamic,

#         "wild_card_probability":
#             dynamic_wild_card_counts.get(
#                 team, 0
#             ) / num_simulations_dynamic

#     })

In [0]:
# dynamic_playoff_results_df = spark.createDataFrame(
#     dynamic_playoff_results
# )

# dynamic_playoff_results_df = dynamic_playoff_results_df.select(
#     "team",
#     "playoff_probability",
#     "division_winner_probability",
#     "wild_card_probability"
# )

In [0]:
# from pyspark.sql.functions import col, round

# dynamic_playoff_results_df = (
#     dynamic_playoff_results_df
#     .withColumn(
#         "playoff_probability",
#         round(col("playoff_probability") * 100, 2)
#     )
#     .withColumn(
#         "division_winner_probability",
#         round(
#             col("division_winner_probability") * 100,
#             2
#         )
#     )
#     .withColumn(
#         "wild_card_probability",
#         round(
#             col("wild_card_probability") * 100,
#             2
#         )
#     )
# )


# display(
#     dynamic_playoff_results_df
#     .orderBy(
#         col("playoff_probability").desc()
#     )
# )